In [1]:
import numpy as np
import matplotlib.pyplot as plt
import cvxpy as cp
from multiprocessing import Process
import multiprocessing
import time

In [2]:
def Greedy_policy(pkt_prob, e_prob, weight_prob, T_val, M, send_end):
    import numpy as np
    import cvxpy as cp
    from itertools import combinations

    # Channel model
    h_bad = 0.5
    h_prob = [h_bad, 1 - h_bad]
    h_vals = [0.1, 1]

    # System limits
    B_max = 2
    rmax  = 1

    # Previous states
    B_prev   = np.zeros(M)
    rem_prev = np.zeros(M)
    wt_start = np.ones(M)

    opt_sol = 0

    for t in range(T_val):
        if t % 500 == 0:
            print("iteration", t)

        # Optimization variables
        P   = cp.Variable(M, nonneg=True)
        rho = cp.Variable(M, nonneg=True)

        # Random realizations
        wt  = np.random.choice([1, 2], size=M, p=[weight_prob, 1 - weight_prob])
        h   = np.random.choice(h_vals, size=M, p=h_prob)
        pkt = np.random.choice([1, 0], size=M, p=[pkt_prob, 1 - pkt_prob])
        E   = np.random.choice([1, 0], size=M, p=[e_prob, 1 - e_prob])

        # Battery update
        B_start = np.minimum(B_prev + E, B_max)

        # Packet state update
        rem_start = np.zeros(M)
        for i in range(M):
            if pkt[i] == 1:
                wt_start[i] = wt[i]
                rem_start[i] = rmax
            else:
                rem_start[i] = rem_prev[i]

        # Objective
        obj = cp.sum([
            wt_start[i] * cp.exp(-(rmax - (rem_start[i] - rho[i])))
            for i in range(M)
        ])

        # Constraints
        constraints = []

        # Per-user constraints
        for i in range(M):
            constraints += [
                P[i]   <= B_start[i],
                rho[i] <= rem_start[i],
                rho[i] <= cp.log(1 + h[i]*P[i]) * cp.inv_pos(cp.log(2)),
                P[i]   <= (cp.exp(rem_start[i]*cp.log(2)) - 1)/h[i]
            ]

        # ✅ Full MAC subset constraints
        users = list(range(M))
        for k in range(1, M+1):
            for S in combinations(users, k):
                idx = list(S)
                constraints.append(
                    cp.sum(rho[idx])
                    <= cp.log(1 + cp.sum(cp.multiply(h[idx], P[idx])))
                       * cp.inv_pos(cp.log(2))
                )

        # Solve
        prob = cp.Problem(cp.Minimize(obj), constraints)
        # optimal_val = prob.solve(solver=cp.SCS, eps=1e-8, verbose=False)
        optimal_val = prob.solve(solver=cp.ECOS, warm_start=True, verbose=False)

        opt_sol += optimal_val

        # State update
        P_val   = np.maximum(P.value, 0)
        rho_val = np.maximum(rho.value, 0)

        B_prev   = np.maximum(B_start - P_val, 0)
        rem_prev = np.maximum(rem_start - rho_val, 0)

        # if t % 500 == 0:
        #     print("res", t, opt_sol / ((t+1)))

    mean_opt = opt_sol / (T_val)
    

    print('final objective', pkt_prob, e_prob, weight_prob, mean_opt)
    print('----------------------------------------------------')
    send_end.send(mean_opt)

In [3]:
lambda_arr = [0.5]
jobs = []
pipe_list = []
for weight_prob in lambda_arr:
    recv_end, send_end = multiprocessing.Pipe(False)
    p = Process(target=Greedy_policy, args = (0.5, 0.5, 0.5, 1001, 2, send_end))
    jobs.append(p)
    pipe_list.append(recv_end)
for process in jobs:
    process.start()
for process in jobs:
    process.join()
final_objective_weight = np.array([x.recv() for x in pipe_list])

iteration 0


iteration 500
iteration 1000
final objective 0.5 0.5 0.5 1.9769885027731715
----------------------------------------------------
